# 00 — Kiểm tra môi trường & rủi ro kỹ thuật cao nhất: cài `mamba-ssm` trên Colab T4

**Chạy notebook này đầu tiên, trên Google Colab với Runtime = GPU (T4).**

Mục tiêu (Tuần 1–2 theo đề cương): xác nhận sớm nhất có thể liệu `mamba-ssm`
có build/chạy được trên T4 hay không, và nếu không, xác nhận phương án dự
phòng (Mamba thuần PyTorch, không cần CUDA kernel tùy biến) hoạt động đúng.

Đây là rủi ro cao nhất của toàn bộ đề tài — nếu mục 3 thất bại hoàn toàn,
cần báo GVHD sớm để điều chỉnh phạm vi.

**Trạng thái (chi tiết đầy đủ ở `docs/notes/mamba_ssm_install_log.md`):**
- Lần 1 (2026-09-14) — Cách 1 (`pip install mamba-ssm` thẳng) **thất bại**
  ngay ở bước build metadata (build isolation không thấy torch cài sẵn).
- Lần 2 (2026-09-14) — Cách 2 (`--no-build-isolation`, build từ source,
  nhánh `main`) **cài thành công**, nhưng import `mamba_ssm` lỗi:
  `AttributeError("attribute '__dict__' of 'type' objects is not writable")`.
- Lần 3 (2026-09-14) — **Đã xác định nguyên nhân thật** (full traceback):
  nhánh `main` của `state-spaces/mamba` (từ v2.3.2, 5/2025) gộp thêm kiến
  trúc **Mamba-3**, và `mamba_ssm/__init__.py` **luôn** import `Mamba3` —
  kéo theo `tilelang` → `tvm`, và bản thân `tvm` có bug đăng ký class
  attribute không tương thích Python 3.12. Đề tài chỉ cần Mamba/S6 cổ
  điển, không cần Mamba-3. **Fix: pin về tag `v2.3.1`** (release cuối cùng
  trước khi có Mamba-3) — đã cập nhật cell cài đặt bên dưới. Fallback
  `selective_scan_ref` (thuần PyTorch) đã xác nhận import thành công ở
  Lần 2 — dù sao đề tài cũng không bị chặn. **Chưa verify lại sau khi pin
  v2.3.1 — cần chạy Lần 4.**

## 1. Kiểm tra GPU & phiên bản torch/CUDA

In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version (torch build):", torch.version.cuda)
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("compute capability:", torch.cuda.get_device_capability(0))

import sys
print("python:", sys.version)

## 2. Cài `mamba-ssm`

Cách 1 (pip thẳng) đã xác nhận thất bại (Lần 1) — bỏ qua. Cách 2
(`--no-build-isolation`, build từ source) cài được nhưng bản `main` kéo theo
Mamba-3/tilelang/tvm lỗi (Lần 2-3) — đã **pin về tag `v2.3.1`** (bản cuối
trước Mamba-3, đủ cho Mamba/S6 cổ điển mà đề tài cần) ở cell bên dưới.

In [ ]:
# Cách 1 — đã thử, THẤT BẠI (xem docs/notes/mamba_ssm_install_log.md, Lần 1).
# Giữ lại đây để tham khảo, không cần chạy lại trừ khi muốn kiểm tra lại từ đầu.

# !pip install -q mamba-ssm causal-conv1d>=1.4.0

In [ ]:
# Cách 2 — build từ source, pin về tag v2.3.1 (bản cuối trước khi có
# Mamba-3/tilelang/tvm — xem log Lần 3). Dùng subprocess để bắt full log lỗi
# (Colab hay thu gọn output dài của lệnh !pip khi lỗi).
import subprocess

def run(cmd):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print("--- STDERR (tail) ---")
        print(result.stderr[-4000:])
    print("return code:", result.returncode)
    return result.returncode == 0

import sys

ok = run([sys.executable, "-m", "pip", "install", "-q", "packaging", "ninja"])
ok = ok and run([
    sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
    "git+https://github.com/Dao-AILab/causal-conv1d",
])
ok = ok and run([
    sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
    "git+https://github.com/state-spaces/mamba@v2.3.1",
])
print("\nCách 2 thành công:" if ok else "\nCách 2 vẫn thất bại — xem STDERR ở trên, ghi lại vào docs/notes/mamba_ssm_install_log.md.", ok)

### 2b. (Tùy chọn) Restart runtime trước khi import

Không còn là nghi vấn chính (nguyên nhân thật đã xác định ở Lần 3: Mamba-3/
tilelang, không phải do torch dispatcher) — nhưng vẫn là thói quen tốt sau
khi cài package build từ source: **Runtime → Restart session** (KHÔNG chọn
"Disconnect and delete runtime"), rồi chạy lại từ mục 1. Nếu bỏ qua bước
này mà cell dưới vẫn lỗi, restart rồi thử lại trước khi kết luận.

In [ ]:
# Kiểm tra import + forward pass nhỏ với CUDA kernel thật.
# In full traceback (không chỉ repr(e)) để chẩn đoán đúng nguyên nhân nếu vẫn lỗi.
import traceback

try:
    from mamba_ssm import Mamba
    m = Mamba(d_model=64, d_state=16, d_conv=4, expand=2).to("cuda")
    x = torch.randn(2, 32, 64, device="cuda")
    y = m(x)
    print("OK — mamba-ssm (CUDA kernel) hoạt động. Output shape:", y.shape)
    MAMBA_CUDA_OK = True
except Exception as e:
    print("THẤT BẠI — mamba-ssm CUDA kernel không chạy được. Full traceback:")
    traceback.print_exc()
    MAMBA_CUDA_OK = False

## 3. Phương án dự phòng: Mamba thuần PyTorch (không cần CUDA kernel)

`mamba_ssm` cung cấp `selective_scan_ref` — cài đặt tham chiếu thuần PyTorch,
**chậm hơn** nhưng đúng về mặt số học và không cần biên dịch bất kỳ CUDA
kernel nào. Đã xác nhận **import thành công** ở Lần 2 — phương án dự phòng
khả dụng dù mục 2 (CUDA kernel thật) còn lỗi.

In [ ]:
try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_ref
    print("OK — selective_scan_ref (thuần PyTorch) import được — có thể dùng làm fallback.")
    FALLBACK_OK = True
except Exception as e:
    print("selective_scan_ref cũng không import được:")
    print(repr(e))
    print("=> Cần cân nhắc phương án khác (vd. cài đặt S4/S6 tối giản từ đầu).")
    FALLBACK_OK = False

## 4. Kết luận rủi ro (ghi lại kết quả để báo cáo GVHD)

In [ ]:
print(f"mamba-ssm CUDA kernel hoạt động trên T4: {MAMBA_CUDA_OK}")
print(f"Fallback thuần PyTorch khả dụng:        {FALLBACK_OK}")
if MAMBA_CUDA_OK:
    print("=> Dùng CUDA kernel thật cho tốc độ train/inference tốt nhất.")
elif FALLBACK_OK:
    print("=> Dùng selective_scan_ref (chậm hơn) — vẫn tiếp tục đề tài đúng kế hoạch,")
    print("   nhưng cần ghi rõ trong báo cáo (Chương 2) và ước lượng lại thời gian train.")
else:
    print("=> RỦI RO NGHIÊM TRỌNG — báo GVHD ngay để điều chỉnh phạm vi đề tài.")

print("\n=> Nhớ copy kết quả (thành công/thất bại + full traceback nếu có) vào")
print("   docs/notes/mamba_ssm_install_log.md trước khi đóng Colab.")

## 5. Cài các thư viện còn lại của pipeline

In [ ]:
!pip install -q librosa soundfile sentencepiece tokenizers datasets jiwer gradio

## 6. (Tuần 3) Thử tải thử VietSuperSpeech — chỉ kiểm tra kết nối, chưa tải toàn bộ

Đã xác nhận (Lần 2): field thật của dataset là
`dict_keys(['audio', 'text', 'duration', 'source'])` — `text` là transcript,
`source` là tên file video gốc (hữu ích khi cần theo dõi/loại nhiễu theo
kênh nguồn). Khớp với giả định trong `src/data/vietsuperspeech_dataset.py`.

In [ ]:
from datasets import load_dataset

# streaming=True để chỉ xem thử vài mẫu, không tải 267 giờ audio ngay bây giờ
ds = load_dataset("thanhnew2001/VietSuperSpeech", split="train", streaming=True)
sample = next(iter(ds))
print(sample.keys())
print({k: v for k, v in sample.items() if k != "audio"})